# 01. Self-Attention 메커니즘

## 학습 목표
- Attention의 직관적 의미 이해: "어디에 집중할 것인가"
- Query, Key, Value의 역할을 검색 비유로 이해
- Scaled Dot-Product Attention을 수식부터 구현까지
- Attention Mask (Padding, Causal) 구현
- Attention Weight 시각화

## 참고 자료
- [Jay Alammar - The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- "Attention Is All You Need" (Vaswani et al., 2017)

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Attention의 직관: "어디에 집중할 것인가"

### 번역 예시로 이해하기

"나는 학생입니다"를 "I am a student"로 번역할 때:

| 영어 단어 | 주목해야 할 한국어 단어 |
|-----------|------------------------|
| I         | **나는** (강하게 집중) |
| am        | **입니다** (동사) |
| student   | **학생** (강하게 집중) |

각 출력 단어를 생성할 때, 입력의 **모든 단어를 보되** 관련 있는 단어에 더 **집중(Attend)** 한다.

이것이 Attention의 핵심 아이디어:
- 모든 입력을 동일하게 보는 것이 아니라
- **현재 작업에 관련 있는 부분에 가중치를 더 준다**

In [ ]:
# 직관적 예시: 단어 간 관련성 점수
# 높은 점수 = 더 많이 집중

source_words = ["나는", "학생", "입니다"]
target_words = ["I", "am", "a", "student"]

# 가상의 attention 점수 (사람이 직관적으로 매긴 것)
attention_scores = torch.tensor([
    [0.9, 0.05, 0.05],   # I → 나는(0.9), 학생(0.05), 입니다(0.05)
    [0.1, 0.1,  0.8],    # am → 나는(0.1), 학생(0.1), 입니다(0.8)
    [0.1, 0.1,  0.8],    # a → (관사, 문법적)
    [0.05, 0.9, 0.05],   # student → 나는(0.05), 학생(0.9), 입니다(0.05)
])

print("Attention 점수 (각 행의 합 = 1):")
for i, tw in enumerate(target_words):
    scores = [f"{source_words[j]}({attention_scores[i,j]:.2f})" for j in range(len(source_words))]
    print(f"  {tw:>8} → {', '.join(scores)}")

---
## 2. Query, Key, Value: 검색 비유

Attention을 **검색 시스템**에 비유하면 이해가 쉽다:

| 개념 | 검색 비유 | Attention에서의 역할 |
|------|-----------|---------------------|
| **Query (Q)** | 검색어 ("맛집") | 지금 찾고 있는 것 |
| **Key (K)** | 문서 제목/태그 | 각 위치의 라벨 |
| **Value (V)** | 문서 내용 | 실제 전달할 정보 |

### 검색 과정
1. **Query와 Key를 비교** → 관련성 점수 계산 (얼마나 매치되는지)
2. **점수를 확률로 변환** → Softmax (합이 1이 되도록)
3. **Value를 가중합** → 관련 높은 정보를 더 많이 가져옴

### 수식으로 보면

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- $QK^T$: Query와 Key의 유사도 (내적)
- $\sqrt{d_k}$: 스케일링 (값이 너무 커지는 것을 방지)
- softmax: 확률 분포로 변환
- $V$: 가중합으로 최종 출력 생성

In [ ]:
# Q, K, V를 검색 비유로 이해하기

# 가상의 문서 데이터베이스
documents = {
    "강남 맛집": "강남역 근처 스시 오마카세, 가격 15만원",
    "홍대 카페": "홍대입구역 분위기 좋은 루프탑 카페",
    "강남 카페": "강남 테헤란로 조용한 작업 카페",
    "판교 맛집": "판교 테크노밸리 점심 맛집 리스트",
}

query = "강남 카페"

# 간단한 유사도: 공통 단어 수
print(f"Query: '{query}'")
print(f"\n각 문서와의 관련성:")
for key, value in documents.items():
    # 공통 글자 수를 유사도로 사용 (간단한 비유)
    common = len(set(query) & set(key))
    print(f"  Key: '{key}' → 유사도: {common}, Value: '{value}'")

print(f"\n→ '강남 카페'라는 Query에 가장 관련 높은 Key의 Value를 더 많이 가져온다!")

---
## 3. Scaled Dot-Product Attention 구현

### Step-by-step 구현

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### 왜 $\sqrt{d_k}$로 나누는가?

내적 값은 차원 $d_k$가 커질수록 분산이 커진다:
- $Q, K$의 각 원소가 평균 0, 분산 1일 때
- $QK^T$의 각 원소의 분산은 $d_k$
- 값이 너무 크면 softmax의 gradient가 거의 0이 됨 (vanishing gradient)
- $\sqrt{d_k}$로 나누면 분산이 1로 정규화됨

In [ ]:
# sqrt(d_k)로 나누는 이유를 실험으로 확인

torch.manual_seed(42)

for d_k in [4, 64, 512]:
    q = torch.randn(1, d_k)  # 평균 0, 분산 1
    k = torch.randn(10, d_k)
    
    # 스케일링 없이
    raw_scores = q @ k.T
    
    # 스케일링 후
    scaled_scores = raw_scores / (d_k ** 0.5)
    
    print(f"d_k = {d_k:>3}:")
    print(f"  스케일링 전 - 평균: {raw_scores.mean():.2f}, 분산: {raw_scores.var():.2f}")
    print(f"  스케일링 후 - 평균: {scaled_scores.mean():.2f}, 분산: {scaled_scores.var():.2f}")
    
    # softmax 결과 비교
    probs_raw = F.softmax(raw_scores, dim=-1)
    probs_scaled = F.softmax(scaled_scores, dim=-1)
    print(f"  softmax(raw)    최대: {probs_raw.max():.4f}, 최소: {probs_raw.min():.6f}")
    print(f"  softmax(scaled) 최대: {probs_scaled.max():.4f}, 최소: {probs_scaled.min():.6f}")
    print(f"  → 스케일링 안 하면 d_k가 클수록 softmax가 one-hot에 가까워짐 (gradient 소실!)")
    print()

In [ ]:
# Step-by-step Scaled Dot-Product Attention 구현

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        Q: Query  (batch, seq_len_q, d_k)
        K: Key    (batch, seq_len_k, d_k)
        V: Value  (batch, seq_len_k, d_v)
        mask: 마스크 (optional)
    
    Returns:
        output: Attention 결과 (batch, seq_len_q, d_v)
        weights: Attention 가중치 (batch, seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    
    # Step 1: Q와 K의 내적 → 유사도 점수
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, seq_q, seq_k)
    print(f"Step 1 - QK^T shape: {scores.shape}")
    
    # Step 2: sqrt(d_k)로 스케일링
    scores = scores / (d_k ** 0.5)
    print(f"Step 2 - Scaled scores 범위: [{scores.min():.2f}, {scores.max():.2f}]")
    
    # Step 3: 마스크 적용 (있으면)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
        print(f"Step 3 - Mask 적용됨")
    
    # Step 4: Softmax → 확률 분포
    weights = F.softmax(scores, dim=-1)
    print(f"Step 4 - Attention weights (각 행의 합 = 1): {weights.sum(dim=-1)}")
    
    # Step 5: Value와 가중합
    output = torch.matmul(weights, V)  # (batch, seq_q, d_v)
    print(f"Step 5 - Output shape: {output.shape}")
    
    return output, weights

In [ ]:
# 간단한 예시로 동작 확인
torch.manual_seed(42)

batch_size = 1
seq_len = 4   # "나는 좋은 학생 입니다" (4 토큰)
d_k = 8       # Key/Query 차원
d_v = 8       # Value 차원

# 임의의 Q, K, V 생성
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_v)

print(f"Q shape: {Q.shape} (batch={batch_size}, seq_len={seq_len}, d_k={d_k})")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print()

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"\n최종 출력 shape: {output.shape}")
print(f"Attention weights:\n{weights.squeeze()}")

### Q, K, V는 어디서 오는가?

실제 Transformer에서는 입력 임베딩 $X$에 학습 가능한 가중치 행렬을 곱해서 만든다:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

- $X$: 입력 임베딩 (seq_len, d_model)
- $W^Q, W^K$: (d_model, d_k)
- $W^V$: (d_model, d_v)

In [ ]:
# 입력 임베딩에서 Q, K, V를 만드는 과정

d_model = 16  # 임베딩 차원
d_k = 8
d_v = 8
seq_len = 4

# 입력 임베딩 (단어 벡터들)
X = torch.randn(1, seq_len, d_model)  # (batch, seq_len, d_model)

# 학습 가능한 가중치 행렬 (Linear layer와 동일)
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_v, bias=False)

# Q, K, V 생성
Q = W_Q(X)  # (1, 4, 8)
K = W_K(X)  # (1, 4, 8)
V = W_V(X)  # (1, 4, 8)

print(f"입력 X shape:  {X.shape} (d_model={d_model})")
print(f"W_Q weight:    {W_Q.weight.shape} → Q shape: {Q.shape} (d_k={d_k})")
print(f"W_K weight:    {W_K.weight.shape} → K shape: {K.shape}")
print(f"W_V weight:    {W_V.weight.shape} → V shape: {V.shape}")
print(f"\n→ 같은 입력 X에서 다른 가중치 행렬로 Q, K, V를 만든다")
print(f"→ Self-Attention: Q, K, V가 모두 같은 시퀀스에서 나옴")

---
## 4. Attention 직접 구현: PyTorch로 step-by-step

이번에는 `nn.Module`로 깔끔하게 구현해보자.

In [ ]:
class SelfAttention(nn.Module):
    """Self-Attention 레이어"""
    
    def __init__(self, d_model, d_k):
        super().__init__()
        self.d_k = d_k
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: 입력 (batch, seq_len, d_model)
            mask: 마스크 (optional)
        Returns:
            output: (batch, seq_len, d_k)
            weights: (batch, seq_len, seq_len)
        """
        Q = self.W_Q(x)  # (batch, seq_len, d_k)
        K = self.W_K(x)
        V = self.W_V(x)
        
        # Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)
        
        return output, weights


# 테스트
torch.manual_seed(42)
d_model = 16
d_k = 8
seq_len = 5

attn = SelfAttention(d_model, d_k)
x = torch.randn(1, seq_len, d_model)

output, weights = attn(x)
print(f"입력 shape:  {x.shape}")
print(f"출력 shape:  {output.shape}")
print(f"가중치 shape: {weights.shape}")
print(f"\nAttention weights (각 행의 합 = 1):")
print(weights.squeeze().detach())

---
## 5. Attention Mask

### 5.1 Padding Mask

배치 처리 시 시퀀스 길이를 맞추기 위해 패딩을 추가한다.
패딩 토큰에는 attention을 주면 안 된다.

```
문장 1: ["나는", "학생", "이다", <PAD>, <PAD>]
문장 2: ["오늘", "날씨", "가", "좋다", <PAD>]
```

In [ ]:
# Padding Mask 구현

def create_padding_mask(seq_lengths, max_len):
    """
    패딩 마스크 생성
    
    Args:
        seq_lengths: 각 시퀀스의 실제 길이 (batch,)
        max_len: 최대 시퀀스 길이
    
    Returns:
        mask: (batch, 1, 1, max_len) - 1이면 attend, 0이면 무시
    """
    batch_size = len(seq_lengths)
    mask = torch.zeros(batch_size, max_len)
    for i, length in enumerate(seq_lengths):
        mask[i, :length] = 1
    return mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, max_len)


# 예시: 배치 2개, 최대 길이 5
seq_lengths = [3, 4]  # 문장1: 3단어, 문장2: 4단어
max_len = 5

padding_mask = create_padding_mask(seq_lengths, max_len)
print(f"Padding mask shape: {padding_mask.shape}")
print(f"문장 1 (길이 3): {padding_mask[0].squeeze()}")
print(f"문장 2 (길이 4): {padding_mask[1].squeeze()}")
print(f"\n→ 0인 위치(패딩)는 attention 점수가 -inf가 되어 softmax 후 0이 됨")

### 5.2 Causal Mask (Look-ahead Mask)

GPT처럼 자기회귀(autoregressive) 모델에서 사용.
현재 위치에서 **미래 토큰을 볼 수 없게** 마스킹한다.

```
토큰 1: [1, 0, 0, 0]  → 자기 자신만
토큰 2: [1, 1, 0, 0]  → 1, 2만
토큰 3: [1, 1, 1, 0]  → 1, 2, 3만
토큰 4: [1, 1, 1, 1]  → 모두 볼 수 있음
```

In [ ]:
# Causal Mask 구현

def create_causal_mask(seq_len):
    """
    Causal (Look-ahead) 마스크 생성
    하삼각 행렬: 현재 위치 이전만 볼 수 있음
    
    Args:
        seq_len: 시퀀스 길이
    Returns:
        mask: (1, 1, seq_len, seq_len)
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)


seq_len = 5
causal_mask = create_causal_mask(seq_len)
print(f"Causal mask (seq_len={seq_len}):")
print(causal_mask.squeeze())
print(f"\n→ 하삼각 행렬: 각 토큰은 자신 이전의 토큰들만 볼 수 있다")

In [ ]:
# Causal Mask 적용 효과 확인

torch.manual_seed(42)
seq_len = 4
d_model = 16
d_k = 8
tokens = ["I", "am", "a", "student"]

attn = SelfAttention(d_model, d_k)
x = torch.randn(1, seq_len, d_model)

# 마스크 없이 (양방향 - BERT 스타일)
output_bi, weights_bi = attn(x)

# Causal 마스크 적용 (단방향 - GPT 스타일)
causal_mask = create_causal_mask(seq_len)
output_causal, weights_causal = attn(x, mask=causal_mask)

print("=== 양방향 Attention (BERT 스타일) ===")
print("각 토큰이 모든 토큰을 볼 수 있음:")
w = weights_bi.squeeze().detach()
for i, t in enumerate(tokens):
    attending = [f"{tokens[j]}({w[i,j]:.2f})" for j in range(len(tokens))]
    print(f"  {t:>8} → {', '.join(attending)}")

print(f"\n=== 단방향 Attention (GPT 스타일) ===")
print("각 토큰이 이전 토큰만 볼 수 있음:")
w = weights_causal.squeeze().detach()
for i, t in enumerate(tokens):
    attending = [f"{tokens[j]}({w[i,j]:.2f})" for j in range(len(tokens))]
    print(f"  {t:>8} → {', '.join(attending)}")

---
## 6. Attention Weight 시각화

Attention weight를 heatmap으로 시각화하면 모델이 어디에 집중하는지 직관적으로 볼 수 있다.

In [ ]:
def plot_attention(weights, src_tokens, tgt_tokens, title="Attention Weights"):
    """
    Attention weight를 heatmap으로 시각화
    
    Args:
        weights: (seq_len_q, seq_len_k) 텐서
        src_tokens: Key 측 토큰 리스트
        tgt_tokens: Query 측 토큰 리스트
        title: 그래프 제목
    """
    fig, ax = plt.subplots(figsize=(6, 5))
    
    w = weights.detach().numpy()
    im = ax.imshow(w, cmap='Blues', aspect='auto')
    
    ax.set_xticks(range(len(src_tokens)))
    ax.set_yticks(range(len(tgt_tokens)))
    ax.set_xticklabels(src_tokens, fontsize=11)
    ax.set_yticklabels(tgt_tokens, fontsize=11)
    ax.set_xlabel("Key (Source)", fontsize=12)
    ax.set_ylabel("Query (Target)", fontsize=12)
    ax.set_title(title, fontsize=13)
    
    # 각 셀에 값 표시
    for i in range(len(tgt_tokens)):
        for j in range(len(src_tokens)):
            ax.text(j, i, f"{w[i, j]:.2f}",
                    ha="center", va="center",
                    color="white" if w[i, j] > 0.5 else "black",
                    fontsize=10)
    
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

In [ ]:
# 양방향 vs 단방향 Attention 시각화 비교

tokens = ["I", "am", "a", "student"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 양방향 (BERT)
w_bi = weights_bi.squeeze().detach().numpy()
im1 = axes[0].imshow(w_bi, cmap='Blues', aspect='auto')
axes[0].set_xticks(range(len(tokens)))
axes[0].set_yticks(range(len(tokens)))
axes[0].set_xticklabels(tokens)
axes[0].set_yticklabels(tokens)
axes[0].set_title("Bidirectional Attention (BERT)", fontsize=13)
axes[0].set_xlabel("Key")
axes[0].set_ylabel("Query")
for i in range(len(tokens)):
    for j in range(len(tokens)):
        axes[0].text(j, i, f"{w_bi[i,j]:.2f}", ha="center", va="center",
                     color="white" if w_bi[i,j] > 0.5 else "black", fontsize=9)
plt.colorbar(im1, ax=axes[0])

# 단방향 (GPT)
w_ca = weights_causal.squeeze().detach().numpy()
im2 = axes[1].imshow(w_ca, cmap='Oranges', aspect='auto')
axes[1].set_xticks(range(len(tokens)))
axes[1].set_yticks(range(len(tokens)))
axes[1].set_xticklabels(tokens)
axes[1].set_yticklabels(tokens)
axes[1].set_title("Causal Attention (GPT)", fontsize=13)
axes[1].set_xlabel("Key")
axes[1].set_ylabel("Query")
for i in range(len(tokens)):
    for j in range(len(tokens)):
        axes[1].text(j, i, f"{w_ca[i,j]:.2f}", ha="center", va="center",
                     color="white" if w_ca[i,j] > 0.5 else "black", fontsize=9)
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("왼쪽 (BERT): 모든 위치 간 attention이 존재")
print("오른쪽 (GPT): 상삼각이 0 → 미래 토큰을 볼 수 없음")

In [ ]:
# 번역 예시: 가상의 Cross-Attention 시각화
# (Encoder-Decoder 모델에서 Decoder가 Encoder 출력에 attend하는 패턴)

src_tokens = ["나는", "좋은", "학생", "입니다"]
tgt_tokens = ["I", "am", "a", "good", "student"]

# 가상의 cross-attention weights (실제로는 학습됨)
cross_attn_weights = torch.tensor([
    [0.85, 0.05, 0.05, 0.05],   # I → 나는
    [0.10, 0.05, 0.05, 0.80],   # am → 입니다
    [0.05, 0.05, 0.05, 0.85],   # a → 입니다 (문법)
    [0.05, 0.80, 0.10, 0.05],   # good → 좋은
    [0.05, 0.10, 0.80, 0.05],   # student → 학생
])

plot_attention(cross_attn_weights, src_tokens, tgt_tokens,
               title="Cross-Attention: Korean → English Translation")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: Scaled Dot-Product Attention 직접 계산

아래 Q, K, V가 주어졌을 때, attention 출력을 **수동으로** (함수 호출 없이) step-by-step 계산하세요.

각 단계의 결과를 출력하세요:
1. $QK^T$ 계산
2. $\sqrt{d_k}$로 나누기
3. Softmax 적용
4. Value와 가중합

In [ ]:
# 간단한 예시 (batch 없이, 2D 텐서로)
Q = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0]])  # (2, 2)

K = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0]])  # (2, 2)

V = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])  # (2, 2)

d_k = Q.size(-1)  # 2

# TODO: Step-by-step으로 attention 계산
# step1 = ...
# step2 = ...
# step3 = ...
# output = ...


### 연습 2: Padding + Causal Mask 동시 적용

아래 시퀀스에 대해 padding mask와 causal mask를 **동시에** 적용하는 combined mask를 만들고,
attention을 수행하세요.

```
시퀀스: ["hello", "world", <PAD>, <PAD>]  (실제 길이 = 2)
```

힌트: 두 마스크를 element-wise 곱하면 됩니다.

In [ ]:
seq_len = 4
actual_length = 2  # "hello", "world"만 실제 토큰

# TODO: 
# 1. Padding mask 생성 (actual_length 이후는 0)
# 2. Causal mask 생성 (하삼각)
# 3. Combined mask = padding_mask * causal_mask
# 4. Combined mask를 시각화 (plt.imshow)
# 5. 이 mask를 적용한 attention weight 확인


---
## 핵심 정리

| 개념 | 설명 | 수식/핵심 |
|------|------|----------|
| Attention | 입력의 어디에 집중할지 학습 | 가중합(weighted sum) |
| Query (Q) | 현재 찾고 있는 것 | $Q = XW^Q$ |
| Key (K) | 각 위치의 라벨 | $K = XW^K$ |
| Value (V) | 실제 전달할 정보 | $V = XW^V$ |
| Scaled Dot-Product | Q와 K의 유사도로 V를 가중합 | $\text{softmax}(QK^T/\sqrt{d_k})V$ |
| $\sqrt{d_k}$ 스케일링 | gradient 소실 방지 | 내적의 분산을 1로 정규화 |
| Padding Mask | 패딩 토큰 무시 | 패딩 위치 = 0 |
| Causal Mask | 미래 토큰 접근 차단 | 하삼각 행렬 |

**다음 노트북**: [02-transformer-from-scratch.ipynb](02-transformer-from-scratch.ipynb) - Transformer 전체 구현